# 03 — Feature Selection (on N02 optimal subsets)

VarFilter → CorrFilter → ExtraTrees importance (top-30) → SFFS (k=1..15)

Per-target selection on the TRAIN SET only (seed=42 split). **Starts from N02's
per-target optimal feature subsets** (saved in `optimal_features.json`), then
further reduces via SFFS wrapper search. Feature NAMES are saved for N04.

**Strategy**: SFFS searches k=1..15 with StandardScaler + ExtraTrees(n=100) —
same pipeline as N02. Best K = lowest CV RMSE. 1-SE rule picks smallest K
within 1 standard error of best. **YS/UTS use 1-SE** (equal/better accuracy
with fewer features). **El uses best-K** — El's CV variance is ~6% of RMSE
(vs ~3% for YS/UTS), making the 1-SE bound proportionally wider; K=5 loses
accuracy, K=7 preserves it.

**Why ExtraTrees for importance ranking**: model-consistent with N02.
n_estimators=200 (not 100) for stable importance — ranking with 100 trees
can fluctuate ±5-10 positions between runs; 200 trees stabilizes the order.
This is a one-time fit per target, not CV, so the extra cost is negligible.

**Evaluation**: StandardScaler + ExtraTrees(n=100), 5-fold CV, R² + RMSE.
Same pipeline as N02. Compares N03 selected vs N02 optimal subset.

**Input**: `data/X_train.npy`, `data/y_train.npy`, `data/optimal_features.json`
**Output**: `data/selected_features.json`

In [1]:
# Load train set + N02 optimal feature sets
import os, json, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')

RNG_SEED = 42; np.random.seed(RNG_SEED)
DATA_DIR = os.path.join('..', 'data')
TARGET_COLS = ['YS', 'UTS', 'El']

# Load N02's per-target optimal feature subsets
with open(os.path.join(DATA_DIR, 'optimal_features.json')) as f:
    opt = json.load(f)
ALL_FEATURES = opt['feature_names_all']

# Load train set
df_train = pd.read_csv(os.path.join(DATA_DIR, 'metal_train_imputed.csv'))
X_train_arr = np.load(os.path.join(DATA_DIR, 'X_train.npy'))
y_train_arr = np.load(os.path.join(DATA_DIR, 'y_train.npy'))
y_df = pd.DataFrame(y_train_arr, columns=TARGET_COLS)

# Raw feature columns (mass fractions + process)
RAW_COLS = ['Si','Fe','Cu','Mn','Mg','Cr','Zn','V','Ti','Zr','Li','Ni','Be','Sc','Tsol','Tage','tage']

print(f'Engineered features: {len(ALL_FEATURES)} | Train samples: {len(df_train)}')
for t in TARGET_COLS:
    label = opt[t]['label']
    n = len(opt[t]['feature_names'])
    print(f'  {t}: N02 optimal = {label} ({n} features)')

Engineered features: 330 | Train samples: 345
  YS: N02 optimal = Raw + B + C (-A,-D) (231 features)
  UTS: N02 optimal = Raw + B + C + D (-A) (276 features)
  El: N02 optimal = Raw + delta(16) only (33 features)


## Run: VarFilter → CorrFilter → ExtraTrees Importance → SFFS (k=1..15)

For each target, start from N02's optimal feature subset. Apply VarFilter +
CorrFilter to remove near-zero-variance and redundant features. Then
**ExtraTrees(n=200)** ranks the survivors by feature importance (200 trees
for stable ranking; one-time fit, not CV). Top-30 enter SFFS wrapper search.

**SFFS pipeline**: StandardScaler + ExtraTrees(n=100) — identical to N02.
Scoring = neg_root_mean_squared_error, 5-fold CV. Per-fold scores used to
compute standard error for the 1-SE rule.

In [2]:
from feature_selection import filter_low_variance, filter_high_correlation
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
import numpy as np

def build_X(flist):
    """Build feature matrix: raw cols from df_train, engineered cols from X_train_arr."""
    raw = [c for c in flist if c in RAW_COLS]
    eng = [c for c in flist if c not in RAW_COLS]
    parts = []
    if raw:
        parts.append(df_train[raw].values)
    if eng:
        idx = [ALL_FEATURES.index(c) for c in eng if c in ALL_FEATURES]
        parts.append(X_train_arr[:, idx])
    return np.column_stack(parts) if len(parts) > 1 else parts[0]

# Strategy: 1-SE rule for YS/UTS, best-K for El.
# El's CV variance is ~6% of RMSE vs ~3% for YS/UTS -- the 1-SE bound is
# proportionally wider for El, making it pick K=5 which loses accuracy.
# Best-K (K=7) preserves El accuracy. This is a data-driven choice, not arbitrary.
STRATEGY = {'YS': '1-SE', 'UTS': '1-SE', 'El': 'best'}
selected_sets = {}

for target_name in TARGET_COLS:
    X_target = build_X(opt[target_name]['feature_names'])
    col_names = opt[target_name]['feature_names']
    n02_label = opt[target_name]['label']
    print(f'\n{target_name}: N02 optimal = {n02_label} ({X_target.shape[1]} features)')

    # Step 1-2: VarFilter + CorrFilter
    X_df_tmp = pd.DataFrame(X_target, columns=col_names)
    X1, cols1 = filter_low_variance(X_df_tmp, threshold=1e-4)
    X2, cols2 = filter_high_correlation(X1, threshold=0.95)
    print(f'  VarFilter: {X_target.shape[1]} -> {len(cols1)}')
    print(f'  CorrFilter: {len(cols1)} -> {len(cols2)}')

    X_arr = X2.values
    yt = y_df[target_name].values

    # Step 3a: ExtraTrees importance ranking -> top-30
    # n_estimators=200 for stable importance (one-time fit, not CV)
    et = ExtraTreesRegressor(n_estimators=200, random_state=RNG_SEED, n_jobs=-1)
    et.fit(X_arr, yt)
    imp_order = np.argsort(et.feature_importances_)[::-1]
    top_k = min(30, len(cols2))
    X_top = X_arr[:, imp_order[:top_k]]
    top_names = [cols2[i] for i in imp_order[:top_k]]

    # Step 3b: SFFS (k=1..15) -- same pipeline as N02
    pipe = Pipeline([('scaler', StandardScaler()),
                     ('model', ExtraTreesRegressor(n_estimators=100, random_state=RNG_SEED, n_jobs=-1))])
    sfs = SFS(pipe, k_features=(1, min(15, top_k)), forward=True, floating=True,
              scoring='neg_root_mean_squared_error', cv=5, n_jobs=-1, verbose=0)
    sfs.fit(X_top, yt)

    # Step 3c: Compute per-K scores, apply strategy
    k_scores = {}
    for k in sfs.subsets_:
        fold_scores = sfs.subsets_[k]['cv_scores']
        rmse = -np.mean(fold_scores)
        se = np.std(fold_scores, ddof=1) / np.sqrt(5)
        k_scores[k] = (rmse, se)

    best_k = min(k_scores, key=lambda k: k_scores[k][0])
    one_se_k = min(k for k, (r, s) in k_scores.items()
                   if r <= k_scores[best_k][0] + k_scores[best_k][1])

    strat = STRATEGY.get(target_name, '1-SE')
    pick_k = one_se_k if strat == '1-SE' else best_k

    sel_local = list(sfs.subsets_[pick_k]['feature_idx'])
    selected = [top_names[i] for i in sel_local]
    selected_sets[target_name] = selected

    print(f'  SFFS: best K={best_k} (RMSE={k_scores[best_k][0]:.2f}), 1-SE K={one_se_k}')
    print(f'  Strategy={strat} -> pick K={pick_k} ({len(selected)} features)')
    print(f'  Features: {", ".join(selected)}')

# Summary
sep = '=' * 60
print(f'\n{sep}')
print(f'  N03 Feature Selection Summary')
print(f'  Strategy: 1-SE for YS/UTS, best-K for El')
print(f'{sep}')
print(f'  {"Target":8s} {"N02 opt":>8s} {"N03 sel":>8s}  {"Strategy":>8s}')
print(f'  {"-"*8} {"-"*8} {"-"*8}  {"-"*8}')
for t in TARGET_COLS:
    n02_n = len(opt[t]['feature_names'])
    n03_n = len(selected_sets[t])
    s = STRATEGY.get(t, '1-SE')
    print(f'  {t:8s} {n02_n:>8d} {n03_n:>8d}  {s:>8s}')


YS: N02 optimal = Raw + B + C (-A,-D) (231 features)
  [VarFilter] Removed 148 near-zero variance features
  [CorrFilter] Removed 37 highly correlated features (r > 0.95)
  VarFilter: 231 -> 83
  CorrFilter: 83 -> 46
  SFFS: best K=15 (RMSE=39.65), 1-SE K=9
  Strategy=1-SE -> pick K=9 (9 features)
  Features: B_cxp_Cu_x_LMP, B_wrt_total_solute, C_prc_Tage_log_tage, B_cxp_Zr_x_Tage, C_prc_Tsol_Tage, B_cxp_Mg_x_Tsol, B_cxp_Fe_x_Tsol, B_cxp_Li_x_log_tage, B_cxp_Cr_x_Tage

UTS: N02 optimal = Raw + B + C + D (-A) (276 features)
  [VarFilter] Removed 153 near-zero variance features
  [CorrFilter] Removed 53 highly correlated features (r > 0.95)
  VarFilter: 276 -> 123
  CorrFilter: 123 -> 70
  SFFS: best K=12 (RMSE=35.43), 1-SE K=5
  Strategy=1-SE -> pick K=5 (5 features)
  Features: D_thm_delta, C_prc_larson_miller, D_el_specific_E, B_wrt_solidus_depression, C_prc_Tsol_Tage

El: N02 optimal = Raw + delta(16) only (33 features)
  [VarFilter] Removed 13 near-zero variance features
  [CorrFil

## Summary

In [3]:
# Print selected feature subsets and save
print('N03 complete. Per-target selected features:')
for t in TARGET_COLS:
    feats = selected_sets[t]
    s = STRATEGY.get(t, '1-SE')
    print(f'\n{t} ({len(feats)} features, strategy={s}):')
    for f in feats:
        print(f'  {f}')

N03 complete. Per-target selected features:

YS (9 features, strategy=1-SE):
  B_cxp_Cu_x_LMP
  B_wrt_total_solute
  C_prc_Tage_log_tage
  B_cxp_Zr_x_Tage
  C_prc_Tsol_Tage
  B_cxp_Mg_x_Tsol
  B_cxp_Fe_x_Tsol
  B_cxp_Li_x_log_tage
  B_cxp_Cr_x_Tage

UTS (5 features, strategy=1-SE):
  D_thm_delta
  C_prc_larson_miller
  D_el_specific_E
  B_wrt_solidus_depression
  C_prc_Tsol_Tage

El (7 features, strategy=best):
  D_delta_Tm_abs_to_Al
  Tage
  tage
  Tsol
  D_delta_IE1_abs_to_Al
  D_delta_chi_to_Al
  Zn


In [4]:
# ============================================================
# Ablation: N02 Optimal vs N03 Selected -- per target, 5-fold CV
# (same pipeline as N02: StandardScaler + ExtraTrees n=100)
# ============================================================
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

print('=' * 80)
print('  ABLATION: N02 Optimal vs N03 Selected')
print('  Pipeline: StandardScaler + ExtraTrees(n=100), 5-fold CV')
print('  Strategy: 1-SE for YS/UTS, best-K for El')
print('  |dRMSE| < 1.5% = equal (within CV noise for n=345 samples)')
print('=' * 80)
print(f'  {"Target":8s} {"Features":>20s} {"n":>4s} {"R2":>8s} {"RMSE":>8s} {"dRMSE":>9s}')
print(f'  {"-"*8} {"-"*20} {"-"*4} {"-"*8} {"-"*8} {"-"*9}')

pipe = Pipeline([('scaler', StandardScaler()),
                 ('model', ExtraTreesRegressor(n_estimators=100, random_state=RNG_SEED))])

for target_name in TARGET_COLS:
    yt = y_df[target_name].values

    X_n02 = build_X(opt[target_name]['feature_names'])
    r2_n02 = np.mean(cross_val_score(pipe, X_n02, yt, cv=5, scoring='r2'))
    rmse_n02 = -np.mean(cross_val_score(pipe, X_n02, yt, cv=5, scoring='neg_root_mean_squared_error'))

    X_sel = build_X(selected_sets[target_name])
    r2_sel = np.mean(cross_val_score(pipe, X_sel, yt, cv=5, scoring='r2'))
    rmse_sel = -np.mean(cross_val_score(pipe, X_sel, yt, cv=5, scoring='neg_root_mean_squared_error'))

    d = (rmse_n02 - rmse_sel) / rmse_n02 * 100
    status = 'better' if d > 1.5 else ('worse' if d < -1.5 else 'equal')

    print(f'  {target_name:8s} {"N02 optimal":>20s} {X_n02.shape[1]:4d} {r2_n02:8.4f} {rmse_n02:8.2f}')
    print(f'  {"":8s} {"N03 selected":>20s} {X_sel.shape[1]:4d} {r2_sel:8.4f} {rmse_sel:8.2f} {d:+8.1f}% ({status})')

print('=' * 80)

with open(os.path.join(DATA_DIR, 'selected_features.json'), 'w') as f:
    json.dump(selected_sets, f, indent=2)
print('Saved selected_features.json')
print('\nNext: 04_model_training.ipynb')

  ABLATION: N02 Optimal vs N03 Selected
  Pipeline: StandardScaler + ExtraTrees(n=100), 5-fold CV
  Strategy: 1-SE for YS/UTS, best-K for El
  |dRMSE| < 1.5% = equal (within CV noise for n=345 samples)
  Target               Features    n       R2     RMSE     dRMSE
  -------- -------------------- ---- -------- -------- ---------
  YS                N02 optimal  231   0.8974    40.76
                   N03 selected    9   0.8965    40.91     -0.4% (equal)
  UTS               N02 optimal  276   0.8892    37.64
                   N03 selected    5   0.8969    36.21     +3.8% (better)
  El                N02 optimal   33   0.7431     3.55
                   N03 selected    7   0.7367     3.58     -0.9% (equal)
Saved selected_features.json

Next: 04_model_training.ipynb
